# NF4 blockpair v1 — prepared research candidate, T4 untested

Original byte-pair decode + block-scale broadcast + aligned packed FP16 stores. Both actual
bitsandbytes0.50.2 decode boundaries; native tiny-M routing, GEMM, casts, and autograd retained.
This is **not** unchanged trial002/004. Historical standalone share failures remain failures.

No GPU run, installation, download, queue submission, or external write was performed in preparation.
This notebook embeds all candidate/driver source. It uses only a **prepared local official checkout,
public data, exact installed packages and public model cache**. It never installs or downloads.

After separate run authorization, supply the three paths below and enable the gate. On Run All it
creates a new staging directory and calls the untouched official paired harness in order
base→solution→solution→base. Every measured subprocess has fresh state/output/compiler caches.
Whole-command time, nvidia-smi peak, public evaluation, exact checks and route receipts are retained.
GCP compares its own paired baseline, not Kaggle's historical79-second screen. Score stays null;
public automatic checks cannot establish hidden quality or final Kaggle/jury admission.


In [ ]:
import os
CONFIG = {
    "authorized": False,  # Set True only after current explicit GPU execution authorization.
    "backend": "GCP",   # Or KAGGLE_T4. Never transplant a baseline between environments.
    "prepared_repo": os.environ.get("SOUP_PREPARED_REPO", "/REPLACE/WITH/PREPARED/OFFICIAL/CHECKOUT"),
    "hf_hub_cache": os.environ.get("HF_HUB_CACHE", "/REPLACE/WITH/PREPARED/PUBLIC/MODEL/CACHE"),
    "output_parent": os.environ.get("SOUP_OUTPUT_ROOT", "/kaggle/working/nf4-blockpair-runs"),
}


## Frozen source and runner
The following cells are generated, not a network bootstrap. Compare their embedded source hashes
to source_manifest.json and candidate.zip before enabling execution. The worker accepts the ZIP,
not this notebook. The submission template intentionally has no measured baseline ID.


In [ ]:
PAYLOAD = {'solution/__init__.py': '"""NF4 blockpair v1: original packed-store decoder; unchanged native gradients."""\n\n\ndef apply():\n    try:\n        from .routes import install\n    except ModuleNotFoundError as exc:\n        if exc.name == "triton":\n            return {"patched": False, "reason": "triton_unavailable"}\n        raise\n    result = install()\n    print("SOUP_NF4_INSTALL " + str(result), flush=True)\n    return result\n', 'solution/packed_nf4.py': '"""Original byte-lane NF4 decoder with block-local scale broadcast.\n\nNot a GEMM or autograd replacement. Geometry is flat 64-weight outer blocks,\n256-scale nested blocks. FP32 scale product and offset addition do not contract.\nThe 16 constants below are public NF4 format data, not kernel source.\n"""\nimport math\nimport torch\nimport triton\nimport triton.language as tl\n\nQWEN_SHAPES = frozenset(((1536, 1536), (256, 1536), (8960, 1536), (1536, 8960)))\n\n\n@triton.jit\ndef _nf4_value(c):\n    # Balanced nibble decisions; each lane decodes one packed byte, not one weight.\n    low0 = tl.where(c == 0, -1.0, -0.6961928009986877)\n    low1 = tl.where(c == 2, -0.5250730514526367, -0.39491748809814453)\n    low2 = tl.where(c == 4, -0.28444138169288635, -0.18477343022823334)\n    low3 = tl.where(c == 6, -0.09105003625154495, 0.0)\n    high0 = tl.where(c == 8, 0.07958029955625534, 0.16093020141124725)\n    high1 = tl.where(c == 10, 0.24611230194568634, 0.33791524171829224)\n    high2 = tl.where(c == 12, 0.44070982933044434, 0.5626170039176941)\n    high3 = tl.where(c == 14, 0.7229568362236023, 1.0)\n    a = tl.where(c < 2, low0, low1)\n    b = tl.where(c < 6, low2, low3)\n    d = tl.where(c < 10, high0, high1)\n    e = tl.where(c < 14, high2, high3)\n    return tl.where(c < 8, tl.where(c < 4, a, b), tl.where(c < 12, d, e))\n\n\n@triton.jit\ndef soup_nf4_blockpair_decode(PACKED, QABS, CODE2, ABS2, OFFSET, OUT,\n                              N: tl.constexpr, SCALES: tl.constexpr,\n                              PACK_HALF: tl.constexpr):\n    # Scales load as a [SCALES] vector, then broadcast across their 32 bytes.\n    # The block index is global: it never resets at a matrix row boundary.\n    block = tl.program_id(0) * SCALES + tl.arange(0, SCALES)\n    valid_block = block * 64 < N\n    q = tl.load(QABS + block, valid_block, other=0).to(tl.int32)\n    c = tl.load(CODE2 + q, valid_block, other=0).to(tl.float32)\n    a = tl.load(ABS2 + block // 256, valid_block, other=0).to(tl.float32)\n    product = c * a\n    scale = product + tl.load(OFFSET).to(tl.float32)\n    pair = block[:, None] * 32 + tl.arange(0, 32)[None, :]\n    even = pair * 2\n    byte = tl.load(PACKED + pair, even < N, other=0).to(tl.uint32)\n    hi = _nf4_value(byte >> 4) * scale[:, None]\n    lo = _nf4_value(byte & 15) * scale[:, None]\n    if PACK_HALF:\n        h = hi.to(tl.float16, fp_downcast_rounding="rtne")\n        l = lo.to(tl.float16, fp_downcast_rounding="rtne")\n        bits = h.to(tl.uint16, bitcast=True).to(tl.uint32)\n        bits |= l.to(tl.uint16, bitcast=True).to(tl.uint32) << 16\n        # CUDA is little-endian. High nibble/even weight occupies low 16 bits.\n        words = OUT.to(tl.pointer_type(tl.uint32))\n        tl.store(words + pair, bits, even + 1 < N)\n        if N % 2:\n            tl.store(OUT + even, h, even == N - 1)\n    else:\n        # FP32 has no intermediate FP16 cast. Also handles unaligned FP16 views.\n        tl.store(OUT + even, hi, even < N)\n        tl.store(OUT + even + 1, lo, even + 1 < N)\n\n\ndef _tensor(t, dtype, count, device):\n    return (type(t) is torch.Tensor and t.layout == torch.strided\n            and t.device == device and t.dtype == dtype and t.numel() == count\n            and t.is_contiguous() and not t.requires_grad)\n\n\ndef raw_supported(B, shape, qabs, code2, abs2, offset, dtype, *, model_only=True):\n    # Metadata-only dispatch. No tensor values, benchmark state, or mutable cache.\n    if (not isinstance(B, torch.Tensor) or not B.is_cuda\n            or B.layout != torch.strided or B.dtype != torch.uint8\n            or B.ndim != 2 or B.shape[1] != 1 or B.shape[0] <= 1\n            or not B.is_contiguous() or B.requires_grad):\n        return False\n    if (not isinstance(shape, (tuple, list, torch.Size)) or len(shape) != 2\n            or any(type(d) is not int or d <= 0 for d in shape)\n            or (model_only and tuple(shape) not in QWEN_SHAPES)\n            or dtype not in (torch.float16, torch.float32)):\n        return False\n    n = math.prod(shape)\n    outer = (n + 63) // 64\n    if B.numel() != (n + 1) // 2:\n        return False\n    specs = ((qabs, torch.uint8, outer), (code2, torch.float32, 256),\n             (abs2, torch.float32, (outer + 255) // 256), (offset, torch.float32, 1))\n    return (all(_tensor(t, d, size, B.device) for t, d, size in specs)\n            and torch.cuda.get_device_capability(B.device) == (7, 5))\n\n\ndef state_supported(B, state):\n    if (state is None or getattr(state, "quant_type", None) != "nf4"\n            or type(getattr(state, "blocksize", None)) is not int or state.blocksize != 64\n            or getattr(state, "nested", False) is not True):\n        return False\n    nested = getattr(state, "state2", None)\n    if (nested is None or getattr(nested, "blocksize", None) != 256\n            or getattr(nested, "dtype", None) != torch.float32\n            or getattr(nested, "nested", False)):\n        return False\n    return raw_supported(B, getattr(state, "shape", None), getattr(state, "absmax", None),\n                         getattr(nested, "code", None), getattr(nested, "absmax", None),\n                         getattr(state, "offset", None), getattr(state, "dtype", None))\n\n\ndef launch_into(B, qabs, code2, abs2, offset, out):\n    """Internal validated launch; accepts CPU tensors only under Triton interpreter tests.\n\n    Production wrappers own their fresh output; no decoded weight or scale cache.\n    The alignment branch is only for an explicitly supplied test/output view.\n    """\n    n = out.numel()\n    packed_half = out.dtype == torch.float16 and out.data_ptr() % 4 == 0\n    soup_nf4_blockpair_decode[(triton.cdiv(n, 64 * 16),)](\n        B, qabs, code2, abs2, offset, out, N=n, SCALES=16,\n        PACK_HALF=packed_half, num_warps=4, enable_fp_fusion=False)\n    return out\n\n\ndef decode(B, shape, qabs, code2, abs2, offset, dtype):\n    # Native lifetime and row-major layout, not a persistent workspace. No stream\n    # ownership/alias hazards or extra 2.44 GiB all-model dequantized-weight cache.\n    with torch.cuda.device(B.device):\n        out = torch.empty(tuple(shape), device=B.device, dtype=dtype)\n        return launch_into(B, qabs, code2, abs2, offset, out)\n', 'solution/routes.py': '"""Two narrow decode boundaries; native dispatch, GEMM and autograd stay intact."""\nfrom functools import wraps\nimport inspect\nimport json\nimport torch\nfrom .packed_nf4 import decode, raw_supported, state_supported\n\n_MARKER = "_soup_nf4_blockpair_v1"\n_COUNTS = {"forward": 0, "functional": 0}\n\n\ndef _activated(route, shape, dtype):\n    _COUNTS[route] += 1\n    n = _COUNTS[route]\n    # Flushed, logarithmic stdout receipts survive a process timeout. This is\n    # activation evidence only; official harness CUDA events establish own share.\n    if n == 1 or n & (n - 1) == 0:\n        print("SOUP_NF4_ROUTE " + json.dumps({"candidate": "nf4_blockpair_v1", "route": route,\n              "calls": n, "shape": list(shape), "dtype": str(dtype),\n              "kernel": "soup_nf4_blockpair_decode"}, sort_keys=True), flush=True)\n\n\ndef forward_supported(A, B, shapeB, absmax, blocksize, quant_type, bias=None,\n                      absmax_8bit=None, absmax_code=None, absmax_offset=None):\n    if (type(A) is not torch.Tensor or not A.is_cuda or A.layout != torch.strided\n            or A.dtype != torch.float16 or A.ndim not in (2, 3)\n            or not A.is_contiguous() or any(d <= 0 for d in A.shape)\n            or type(blocksize) is not int or blocksize != 64 or quant_type != "nf4"):\n        return False\n    if not raw_supported(B, shapeB, absmax_8bit, absmax_code, absmax, absmax_offset, A.dtype):\n        return False\n    if B.device != A.device or A.shape[-1] != shapeB[1]:\n        return False\n    # Bias participates in unchanged F.linear. Native MatMul4Bit owns bias grad.\n    if bias is not None and (type(bias) is not torch.Tensor or bias.device != A.device\n                            or bias.dtype != A.dtype or bias.shape != (shapeB[0],)\n                            or bias.layout != torch.strided or not bias.is_contiguous()):\n        return False\n    return True\n\n\ndef forward_wrapper(original):\n    @wraps(original)\n    def wrapped(A, B, shapeB, absmax, blocksize, quant_type, bias=None,\n                absmax_8bit=None, absmax_code=None, absmax_offset=None):\n        if not forward_supported(A, B, shapeB, absmax, blocksize, quant_type, bias,\n                                 absmax_8bit, absmax_code, absmax_offset):\n            return original(A, B, shapeB, absmax, blocksize, quant_type, bias,\n                            absmax_8bit=absmax_8bit, absmax_code=absmax_code,\n                            absmax_offset=absmax_offset)\n        weight = decode(B, shapeB, absmax_8bit, absmax_code, absmax, absmax_offset, A.dtype)\n        _activated("forward", shapeB, A.dtype)\n        return torch.nn.functional.linear(A, weight, bias)\n    setattr(wrapped, _MARKER, True)\n    return wrapped\n\n\ndef functional_wrapper(original):\n    @wraps(original)\n    def wrapped(A, quant_state=None, absmax=None, out=None, blocksize=None, quant_type="fp4"):\n        if out is not None or not state_supported(A, quant_state):\n            return original(A, quant_state=quant_state, absmax=absmax, out=out,\n                            blocksize=blocksize, quant_type=quant_type)\n        q = quant_state\n        result = decode(A, q.shape, q.absmax, q.state2.code, q.state2.absmax, q.offset, q.dtype)\n        _activated("functional", q.shape, q.dtype)\n        return result\n    setattr(wrapped, _MARKER, True)\n    return wrapped\n\n\ndef install():\n    import bitsandbytes as bnb\n    import bitsandbytes.functional as functional\n    if bnb.__version__ != "0.50.2":\n        return {"patched": False, "reason": "requires_bitsandbytes_0.50.2"}\n    if not torch.cuda.is_available() or torch.cuda.get_device_capability() != (7, 5):\n        return {"patched": False, "reason": "requires_sm75_cuda"}\n    import bitsandbytes.backends.cuda.ops as cuda_ops\n    fwd, fun = cuda_ops._dequant_linear_fallback, functional.dequantize_4bit\n    if getattr(fwd, _MARKER, False) and getattr(fun, _MARKER, False):\n        return {"patched": True, "already_patched": True}\n    expected = ((fwd, "bitsandbytes.backends.cuda.ops", "_dequant_linear_fallback",\n                 ("A", "B", "shapeB", "absmax", "blocksize", "quant_type", "bias", "absmax_8bit", "absmax_code", "absmax_offset")),\n                (fun, "bitsandbytes.functional", "dequantize_4bit",\n                 ("A", "quant_state", "absmax", "out", "blocksize", "quant_type")))\n    for fn, module, name, params in expected:\n        if (getattr(fn, "__module__", None) != module or getattr(fn, "__name__", None) != name\n                or hasattr(fn, "__wrapped__") or tuple(inspect.signature(fn).parameters) != params):\n            return {"patched": False, "reason": "unreviewed_decode_boundary"}\n    # Check both before changing either. Do not replace MatMul4Bit or tiny-M routing.\n    cuda_ops._dequant_linear_fallback = forward_wrapper(fwd)\n    functional.dequantize_4bit = functional_wrapper(fun)\n    return {"patched": True, "already_patched": False, "routes": ["forward", "functional"]}\n', 'support/check_kaggle_environment.py': '#!/usr/bin/env python3\n"""Fail-closed package compatibility check; GPU inventory only with explicit --gpu."""\nimport argparse,hashlib,importlib.metadata,json,os,platform,subprocess,sys\nfrom pathlib import Path\nEXPECTED={\'torch\':\'2.11.0+cu128\',\'triton\':\'3.6.0\',\'soup-cli\':\'0.77.0\',\'transformers\':\'5.16.1\',\'peft\':\'0.20.0\',\'trl\':\'0.29.1\',\'bitsandbytes\':\'0.50.2\',\'accelerate\':\'1.14.0\',\'datasets\':\'4.8.5\'}\nCOMMIT=\'c7c2fcd15972ce8b898d8d3ff732ee192c817ceb\'\ndef inspect(gpu=False,repo=None,require_four_cpus=False):\n errors=[];versions={}\n for name,wanted in EXPECTED.items():\n  try:actual=importlib.metadata.version(name)\n  except importlib.metadata.PackageNotFoundError:actual=None\n  versions[name]=actual\n  if actual!=wanted:errors.append(f\'{name}: expected {wanted}, observed {actual}\')\n if sys.version_info[:3]!=(3,13,15):errors.append(\'Exact Python3.13.15 required\')\n affinity=sorted(os.sched_getaffinity(0)) if hasattr(os,\'sched_getaffinity\') else None\n if require_four_cpus and (affinity is None or len(affinity)!=4):errors.append(\'Process affinity must be exactly four logical CPUs; this does not equate CPU hardware\')\n out={\'python\':platform.python_version(),\'executable\':sys.executable,\'platform\':platform.platform(),\'packages\':versions,\'expected_packages\':EXPECTED,\'cpu_affinity\':affinity,\'gpu_checked\':gpu,\'official_commit_expected\':COMMIT,\'errors\':errors,\'equivalent_hardware_claim\':False}\n if repo is not None:\n  root=Path(repo).resolve();r=subprocess.run([\'git\',\'-C\',str(root),\'rev-parse\',\'HEAD\'],capture_output=True,text=True)\n  out[\'repo_commit\']=r.stdout.strip()\n  if r.returncode or out[\'repo_commit\']!=COMMIT:errors.append(\'Official repository commit mismatch\')\n  r=subprocess.run([\'git\',\'-C\',str(root),\'status\',\'--porcelain\',\'--untracked-files=all\',\'--\',\'harness\',\'task\',\'tests\'],capture_output=True,text=True);out[\'frozen_paths_status\']=r.stdout\n  if r.returncode or r.stdout:errors.append(\'Official harness/task/tests must be unchanged, including untracked files\')\n if gpu:\n  try:\n   import torch\n   out[\'cuda_version\']=torch.version.cuda;out[\'torch_git_version\']=torch.version.git_version\n   if torch.version.cuda!=\'12.8\':errors.append(\'Torch compiled CUDA12.8 required\')\n   out[\'visible_device_count\']=torch.cuda.device_count();out[\'visible_device_names\']=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]\n   if out[\'visible_device_count\']!=1 or out[\'visible_device_names\']!=[\'Tesla T4\']:errors.append(\'Exactly one visible Tesla T4 required\')\n   r=subprocess.run([\'nvidia-smi\',\'--query-gpu=name,driver_version,memory.total\',\'--format=csv,noheader\'],capture_output=True,text=True,timeout=10);out[\'nvidia_smi_inventory\']=r.stdout.strip()\n   if r.returncode:errors.append(\'nvidia-smi inventory failed; native memory sampling unavailable\')\n  except Exception as e:errors.append(\'GPU inventory failed: \'+type(e).__name__)\n out[\'passed\']=not errors;return out\nif __name__==\'__main__\':\n p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--out\',required=True);p.add_argument(\'--gpu\',action=\'store_true\');p.add_argument(\'--repo\');p.add_argument(\'--require-four-cpus\',action=\'store_true\');a=p.parse_args();result=inspect(a.gpu,a.repo,a.require_four_cpus);Path(a.out).write_text(json.dumps(result,indent=2)+\'\\n\');print(json.dumps({\'passed\':result[\'passed\'],\'errors\':result[\'errors\'],\'report\':a.out}));raise SystemExit(not result[\'passed\'])\n', 'support/gpu_preflight.py': '"""Future authorized exact T4 preflight. Never imported by solution.apply()."""\nimport argparse\nfrom contextlib import contextmanager\nimport json\nfrom pathlib import Path\nimport sys\nimport torch\nimport bitsandbytes as bnb\nimport bitsandbytes.functional as F\nimport bitsandbytes.backends.cuda.ops as ops\nfrom bitsandbytes.autograd._functions import MatMul4Bit\nfrom solution import routes\nfrom solution.packed_nf4 import decode, state_supported\n\n\n@contextmanager\ndef boundaries(forward, functional):\n    old = ops._dequant_linear_fallback, F.dequantize_4bit\n    ops._dequant_linear_fallback, F.dequantize_4bit = forward, functional\n    try:\n        yield\n    finally:\n        ops._dequant_linear_fallback, F.dequantize_4bit = old\n\n\ndef exact(actual, expected, label):\n    assert actual.shape == expected.shape and actual.dtype == expected.dtype, label\n    assert actual.stride() == expected.stride(), label + ": stride"\n    finite = torch.isfinite(actual) & torch.isfinite(expected)\n    assert torch.equal(torch.isnan(actual), torch.isnan(expected)), label + ": NaN"\n    assert torch.equal(torch.isposinf(actual), torch.isposinf(expected)), label + ": +Inf"\n    assert torch.equal(torch.isneginf(actual), torch.isneginf(expected)), label + ": -Inf"\n    bits = torch.int16 if actual.dtype == torch.float16 else torch.int32\n    assert torch.equal(actual[finite].view(bits), expected[finite].view(bits)), label + ": bits"\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--out", required=True)\n    args = parser.parse_args()\n    assert bnb.__version__ == "0.50.2"\n    assert torch.cuda.device_count() == 1 and torch.cuda.get_device_name(0) == "Tesla T4"\n    assert torch.cuda.get_device_capability(0) == (7, 5)\n    torch.manual_seed(20261030)\n    native_fwd, native_fun, native_back = ops._dequant_linear_fallback, F.dequantize_4bit, MatMul4Bit.backward\n    own_fwd, own_fun = routes.forward_wrapper(native_fwd), routes.functional_wrapper(native_fun)\n    results = []\n    shapes = {(1536, 1536): (4, 8, 65, 356), (256, 1536): (4, 356),\n              (8960, 1536): (128, 129, 356), (1536, 8960): (64, 65, 356)}\n    for shape, ms in shapes.items():\n        for dtype in (torch.float16, torch.float32):\n            dense = torch.randn(shape, device="cuda", dtype=dtype) * .03\n            packed, state = F.quantize_4bit(dense, compress_statistics=True, quant_type="nf4", blocksize=64)\n            assert state_supported(packed, state)\n            before = [t.clone() for t in (packed, state.absmax, state.state2.code, state.state2.absmax, state.offset)]\n            actual = own_fun(packed, state)\n            reference = native_fun(packed, state)\n            exact(actual, reference, "decode")\n            del actual, reference, dense\n            for m in ms:\n                x = torch.randn((m, shape[1]), device="cuda", dtype=torch.float16)\n                cotangent = torch.randn((m, shape[0]), device="cuda", dtype=torch.float16) * 8\n                bias = torch.randn(shape[0], device="cuda", dtype=torch.float16) * .02\n                observations = []\n                # Native operator decides fused-vs-fallback independently each time.\n                for fwd, fun in ((native_fwd, native_fun), (own_fwd, own_fun)):\n                    a = x.clone().requires_grad_(True)\n                    b = bias.clone().requires_grad_(True)\n                    with boundaries(fwd, fun):\n                        y = bnb.matmul_4bit(a, packed.t(), quant_state=state, bias=b)\n                        grads = torch.autograd.grad(y, (a, b), cotangent)\n                    observations.append((y.detach(), grads[0], grads[1]))\n                for i, label in enumerate(("output", "input_gradient", "bias_gradient")):\n                    exact(observations[1][i], observations[0][i], label)\n                results.append({"shape": list(shape), "state_dtype": str(dtype), "M": m,\n                                "decode_output_input_gradient_bias_gradient_exact": True})\n                del observations, x, cotangent, bias, y, grads, a, b\n            for original, saved in zip((packed, state.absmax, state.state2.code, state.state2.absmax, state.offset), before):\n                assert torch.equal(original.reshape(-1).view(torch.uint8), saved.reshape(-1).view(torch.uint8))\n            del packed, state, before\n    assert MatMul4Bit.backward is native_back\n    assert routes._COUNTS["forward"] > 0 and routes._COUNTS["functional"] > 0\n    result = {"status": "GPU_PREFLIGHT_PASSED", "cases": results, "native_backward_identity_preserved": True,\n              "routes": dict(routes._COUNTS), "official_admission": "UNKNOWN", "score": None,\n              "limits": "Fixture parity only: no full-task speed/share/quality result"}\n    Path(args.out).write_text(json.dumps(result, indent=2) + "\\n")\n    print(json.dumps(result, indent=2))\n\nif __name__ == "__main__":\n    main()\n', 'support/prepare_asset_manifest.py': '#!/usr/bin/env python3\n"""Read/hash only the selected public model cache + public task files; no download."""\nimport argparse,hashlib,json,re\nfrom pathlib import Path\nMODEL=\'models--Qwen--Qwen2.5-1.5B-Instruct\'\nREVISION=\'989aa7980e4cf806f80c7fef2b1adb7bc71aa306\'\nDATA={\'data/train.jsonl\':\'93d036d912bd81c337dc29893e13a76f36161d96f896ef4e02492e2f1a21a0dd\',\'data/eval_public.jsonl\':\'b7f85b78ba3186dce4deab7cfb8bf8dcf7998e1517c5749ac0f5005886c6f70e\'}\ndef sha(path):\n h=hashlib.sha256()\n with path.open(\'rb\') as f:\n  for b in iter(lambda:f.read(4*1024*1024),b\'\'):h.update(b)\n return h.hexdigest()\ndef build(cache,repo,revision=REVISION):\n cache=Path(cache).resolve();repo=Path(repo).resolve()\n if not re.fullmatch(\'[0-9a-f]{40}\',revision):raise ValueError(\'Revision must be40hex\')\n root=cache/MODEL\n if root.is_symlink() or not root.is_dir():raise ValueError(\'Selected model cache missing or symlinked\')\n for directory in (root/\'refs\',root/\'snapshots\',root/\'blobs\'):\n  if directory.is_symlink() or not directory.is_dir():raise ValueError(\'Model metadata/blob directories must be real local directories\')\n ref=root/\'refs/main\'\n if ref.is_symlink() or not ref.is_file() or ref.read_text().strip()!=revision:raise ValueError(\'refs/main must identify exact pinned revision; no silent revision substitution\')\n snapshot=root/\'snapshots\'/revision\n if snapshot.is_symlink() or not snapshot.is_dir():raise ValueError(\'Snapshot directory absent; restore validated retained links separately, do not guess\')\n files={str(ref.relative_to(cache)):sha(ref)};memo={}\n for p in sorted(snapshot.rglob(\'*\')):\n  if p.is_dir():\n   if p.is_symlink():raise ValueError(\'Linked snapshot directory rejected\')\n   continue\n  resolved=p.resolve(strict=True)\n  if not resolved.is_relative_to((root/\'blobs\').resolve()) or not resolved.is_file():raise ValueError(\'Snapshot target must remain inside selected model blobs\')\n  if resolved not in memo:memo[resolved]=sha(resolved)\n  files[str(p.relative_to(cache))]=memo[resolved]\n if len(files)<2:raise ValueError(\'Empty model snapshot\')\n data={}\n for name,wanted in DATA.items():\n  p=repo/name\n  if not p.is_file() or p.is_symlink() or not p.resolve().is_relative_to(repo) or sha(p)!=wanted:raise ValueError(\'Public data hash mismatch: \'+name)\n  data[name]=wanted\n return {\'model_id\':\'Qwen/Qwen2.5-1.5B-Instruct\',\'revision\':revision,\'files\':files},data\nif __name__==\'__main__\':\n p=argparse.ArgumentParser(description=__doc__);p.add_argument(\'--cache\',required=True);p.add_argument(\'--repo\',required=True);p.add_argument(\'--out\',required=True);a=p.parse_args();manifest,data=build(a.cache,a.repo);out=Path(a.out)\n with out.open(\'x\') as f:json.dump(manifest,f,indent=2);f.write(\'\\n\')\n print(json.dumps({\'model_cache_manifest\':str(out),\'model_cache_manifest_sha256\':sha(out),\'data_files\':data},indent=2))\n'}
FROZEN_MANIFEST = {'harness/_train.py': '45a62642dceab32cf3efcda504de87a9c25ea28b119b1806f56d14ff1146a89d', 'harness/check_rules.py': '879ab1861674f6e714eadd00aaf044235d98526dd75fab1035deec3401121ef5', 'harness/common.py': 'c83066f24f266dab62ebcc9c2d44f5a65902760b53da912e65ebfe511579c2db', 'harness/eval_adapter.py': '47cddbb8e4579c69f9535b099ff2371b1c3af08bfbd93d14bf79aadd74d38482', 'harness/example_reference.py': '6a11055f4a1822e53971c228c6e3e3aa6b88edd9eb6db81a388e26a70a030ceb', 'harness/run_task.py': 'd935a8959f348977313f2eef36870ff2699b9ba013a7978918ff35e148968a79', 'harness/score.py': '731c9b644dcc4069b0b069616f438064d1f7f376a202feb15201fd910ec688c5', 'task/prepare_data.py': 'a8543f11a55aff80dbdc714a0e62391244a5296616eea2e34dc2eb3ff3ec18f7', 'task/task.yaml': 'dd70c739fcd143e06aef3bb1efcc9386f682bf1de26f30e08409d86a97ff47ce', 'tests/conftest.py': 'a82eabab468ea0ef1e003cccd7c3827ce2f4d00d09c32948a34fb48221f7337a', 'tests/test_callback.py': '1919303a3670c0a33abbfce9b33eab0913bb64a3a21c3312d86531c32b5bf490', 'tests/test_config.py': 'db808c44470d144b7cf18b7329492f41f76e37cfc24306e7cba0488bca9d787c', 'tests/test_formats.py': '5d656ed50c59f794e934473d459ea85635f892ba028e192414474e9b5d65919e', 'tests/test_loader.py': '3dad1bea2d33aab97e52b7cd44888d768bc9519af44fbe6cf3e96c260026632f', 'tests/test_trainer_init.py': 'b7608d62a40983bf7c9e5b270e0e527b641a837e462d6df2909fec88f607e821'}


In [ ]:
"""Prepared-assets-only paired runner. This is orchestration, not solution code.

No downloads, installs, queue submissions, edits of prepared checkout, or scoring.
Calling run requires current authorization and an idle single-visible-T4 session.
"""
import hashlib
import fcntl
import io
import json
import math
import os
from pathlib import Path
import shutil
import signal
import subprocess
import sys
import tarfile
import time
import uuid

COMMIT = "c7c2fcd15972ce8b898d8d3ff732ee192c817ceb"
QUARANTINE_PATH = Path("/tmp/soup-gpu0.lock.blocked")
GPU_LOCK_PATH = Path("/tmp/soup-gpu0.lock")
DATA = {"data/train.jsonl": "93d036d912bd81c337dc29893e13a76f36161d96f896ef4e02492e2f1a21a0dd",
        "data/eval_public.jsonl": "b7f85b78ba3186dce4deab7cfb8bf8dcf7998e1517c5749ac0f5005886c6f70e"}


def sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(4 * 1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def dump(path, value):
    Path(path).write_text(json.dumps(value, indent=2, allow_nan=False) + "\n")


def process_record(pid):
    try:
        fields = (Path("/proc") / str(pid) / "stat").read_text().rsplit(")", 1)[1].split()
        return {"parent": int(fields[1]), "state": fields[0], "birth": int(fields[19])}
    except (FileNotFoundError, ProcessLookupError):
        return None


def remember_owned(known):
    snapshot = {}
    for stat in Path("/proc").glob("[0-9]*/stat"):
        pid = int(stat.parent.name)
        record = process_record(pid)
        if record is not None:
            snapshot[pid] = record
    alive = {pid for pid, birth in known.items()
             if pid in snapshot and snapshot[pid]["birth"] == birth}
    while True:
        new = {pid for pid, record in snapshot.items() if record["parent"] in alive}
        if new <= alive:
            break
        alive |= new
    for pid in alive:
        known[pid] = snapshot[pid]["birth"]
    return known


def signal_owned(pid, birth, sig):
    # Open a stable process handle, then verify its /proc birth identity. A PID
    # reused after that check cannot redirect the signal to a different process.
    try:
        fd = os.pidfd_open(pid, 0)
    except ProcessLookupError:
        return
    try:
        record = process_record(pid)
        if record is None or record["birth"] != birth or record["state"] == "Z":
            return
        try:
            signal.pidfd_send_signal(fd, sig)
        except ProcessLookupError:
            pass
    finally:
        os.close(fd)


def terminate_owned_tree(proc, known):
    # Freeze each birth-identified process, rescan for children created before its
    # stop, and repeat until no new identities remain. Never kill a shared group.
    while True:
        remember_owned(known)
        before = dict(known)
        for pid, birth in known.items():
            signal_owned(pid, birth, signal.SIGSTOP)
        remember_owned(known)
        if known == before:
            break
    for pid, birth in sorted(known.items(), reverse=True):
        signal_owned(pid, birth, signal.SIGKILL)
    try:
        proc.wait(timeout=30)
    except subprocess.TimeoutExpired:
        pass
    # SIGKILL delivery is asynchronous. Wait for observed identities to become
    # absent/zombie rather than mistaking a still-transitioning task for a leak.
    cleanup_deadline = time.monotonic() + 5
    while True:
        live = []
        for pid, birth in known.items():
            record = process_record(pid)
            if record is not None and record["birth"] == birth and record["state"] != "Z":
                live.append(pid)
        if not live or time.monotonic() >= cleanup_deadline:
            break
        time.sleep(0.01)
    if live:
        # Never turn uncertain cleanup into permission for a subsequent notebook.
        QUARANTINE_PATH.write_text(json.dumps({"reason": "owned_cleanup_uncertain", "pids": live}))
    return sorted(known), not live


def command(cmd, cwd, env, log, timeout):
    start = time.perf_counter()
    timed_out = False
    cleanup_pids = []
    cleanup_confirmed = None
    interrupted = None
    known = {}
    p = None
    with Path(log).open("x") as stream:
        try:
            p = subprocess.Popen(cmd, cwd=cwd, env=env, stdout=stream,
                                 stderr=subprocess.STDOUT, start_new_session=True)
            root = process_record(p.pid)
            if root is not None:
                known[p.pid] = root["birth"]
            deadline = time.monotonic() + timeout
            while True:
                remember_owned(known)
                remaining = deadline - time.monotonic()
                if remaining <= 0:
                    raise subprocess.TimeoutExpired(cmd, timeout)
                try:
                    code = p.wait(timeout=min(remaining, 0.5))
                    break
                except subprocess.TimeoutExpired:
                    if time.monotonic() >= deadline:
                        raise
        except subprocess.TimeoutExpired:
            timed_out = True
            cleanup_pids, cleanup_confirmed = terminate_owned_tree(p, known)
            code = p.returncode
        except BaseException as exc:
            interrupted = type(exc).__name__
            if p is not None:
                if p.pid not in known:
                    root = process_record(p.pid)
                    if root is not None:
                        known[p.pid] = root["birth"]
                cleanup_pids, cleanup_confirmed = terminate_owned_tree(p, known)
            # A constructor interruption may leave no returned PID handle; its
            # evidence stays unknown and quarantine is mandatory, never retried.
            raise
        finally:
            record = {"command": cmd, "exit_code": p.returncode if p is not None else None,
                      "outer_timeout": timed_out, "interrupted": interrupted,
                      "outer_wall_s": time.perf_counter() - start, "log": str(log),
                      "timeout_owned_pids_signalled": cleanup_pids,
                      "cleanup_confirmed": cleanup_confirmed}
            if timed_out or interrupted is not None:
                # A process census cannot prove that no descendant escaped before
                # it was observed. Quarantine this lane until explicit operator
                # reconciliation; do not equate observed cleanup with safe retry.
                QUARANTINE_PATH.write_text(json.dumps({"reason": "outer_attempt_interrupted_or_timed_out",
                    "log": str(log), "observed_cleanup_confirmed": cleanup_confirmed,
                    "requires_operator_reconciliation": True}))
            dump(str(log) + ".receipt.json", record)
    if cleanup_confirmed is False:
        raise RuntimeError("Owned process cleanup uncertain; GPU lock blocked marker written")
    return record


def run(config, payload, frozen_manifest):
    if config.get("authorized") is not True:
        raise RuntimeError("Prepared only. Explicit current GPU execution authorization is required.")
    if not hasattr(os, "pidfd_open") or not hasattr(signal, "pidfd_send_signal"):
        raise RuntimeError("Linux pidfd cleanup support is required before any GPU work")
    # Check kernel support now, not for the first time during an interruption.
    probe_fd = os.pidfd_open(os.getpid(), 0)
    os.close(probe_fd)
    # Canonical cross-notebook lock: no second GPU lane can overlap this pair.
    if QUARANTINE_PATH.exists():
        raise RuntimeError("GPU0 cleanup requires reconciliation; blocked marker exists")
    with GPU_LOCK_PATH.open("a+") as lock:
        try:
            fcntl.flock(lock.fileno(), fcntl.LOCK_EX | fcntl.LOCK_NB)
        except BlockingIOError as exc:
            raise RuntimeError("Another reviewed notebook owns GPU0; not started") from exc
        previous_term = signal.getsignal(signal.SIGTERM)
        owner = uuid.uuid4().hex
        try:
            if QUARANTINE_PATH.exists():
                raise RuntimeError("GPU0 cleanup requires reconciliation; blocked marker exists")
            # Crash-first marker: SIGKILL/kernel death cannot run Python cleanup.
            # It persists after flock is released by the OS, blocking every next
            # cooperating notebook until explicit operator reconciliation.
            with QUARANTINE_PATH.open("x") as marker:
                json.dump({"owner": owner, "pid": os.getpid(), "state": "active_attempt",
                           "requires_operator_reconciliation_if_abandoned": True}, marker)
                marker.flush()
                os.fsync(marker.fileno())
            def terminate(signum, frame):
                raise SystemExit(128 + signum)
            signal.signal(signal.SIGTERM, terminate)
            result = _run_locked(config, payload, frozen_manifest)
            # Normal complete return only. Timeout/interrupt writes a marker with
            # no matching owner, and any exception leaves the active marker.
            marker = json.loads(QUARANTINE_PATH.read_text())
            if marker.get("owner") != owner:
                raise RuntimeError("Attempt cleanup quarantined; operator reconciliation required")
            QUARANTINE_PATH.unlink()
            return result
        finally:
            signal.signal(signal.SIGTERM, previous_term)
            fcntl.flock(lock.fileno(), fcntl.LOCK_UN)


def _run_locked(config, payload, frozen_manifest):
    if config.get("authorized") is not True:
        raise RuntimeError("Prepared only. Explicit current GPU execution authorization is required.")
    if config.get("backend") not in ("KAGGLE_T4", "GCP"):
        raise ValueError("Declare KAGGLE_T4 or GCP; each run measures its own same-environment baseline")
    prepared = Path(config["prepared_repo"]).expanduser().resolve(strict=True)
    cache = Path(config["hf_hub_cache"]).expanduser().resolve(strict=True)
    output_parent = Path(config["output_parent"]).expanduser().resolve()
    output_parent.mkdir(parents=True, exist_ok=True)
    runroot = output_parent / ("nf4_blockpair_" + uuid.uuid4().hex)
    runroot.mkdir()
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0", CUDA_DEVICE_ORDER="PCI_BUS_ID",
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", PYTHONUNBUFFERED="1",
               HF_HUB_CACHE=str(cache), HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
               TOKENIZERS_PARALLELISM="false", SOUP_NO_AUDIT_LOG="1")
    env.pop("TRITON_INTERPRET", None)
    # No implicit GPU use in the notebook kernel. All GPU work is in child processes.
    helper = runroot / "support"
    helper.mkdir()
    for name, contents in payload.items():
        if name.startswith("support/"):
            dest = runroot / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(contents)
    commit = subprocess.check_output(["git", "-C", str(prepared), "rev-parse", "HEAD"], text=True).strip()
    if commit != COMMIT:
        raise RuntimeError("Prepared checkout is not pinned official starter")
    for rel, wanted in frozen_manifest.items():
        if sha(prepared / rel) != wanted:
            raise RuntimeError("Frozen official source mismatch: " + rel)
    for rel, wanted in DATA.items():
        if sha(prepared / rel) != wanted:
            raise RuntimeError("Fixed public data mismatch: " + rel)
    # Check exact public cache revision/bytes, never fetch a model or prepare data.
    assets = command([sys.executable, str(helper / "prepare_asset_manifest.py"), "--cache", str(cache),
                      "--repo", str(prepared), "--out", str(runroot / "model_manifest.json")],
                     prepared, env, runroot / "assets.log", 600)
    if assets["exit_code"] != 0:
        raise RuntimeError("Prepared assets are incomplete; see " + assets["log"])
    # Build an isolated source copy from local pinned git objects, not a network clone.
    raw = subprocess.check_output(["git", "-C", str(prepared), "archive", COMMIT])
    stage = runroot / "repo"
    stage.mkdir()
    with tarfile.open(fileobj=io.BytesIO(raw), mode="r:") as archive:
        for member in archive.getmembers():
            dest = (stage / member.name).resolve()
            if not dest.is_relative_to(stage) or member.issym() or member.islnk():
                raise RuntimeError("Unexpected archive path/link")
        archive.extractall(stage, filter="data")
    shutil.rmtree(stage / "solution")
    (stage / "solution").mkdir()
    for name, contents in payload.items():
        if name.startswith("solution/"):
            dest = stage / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            dest.write_text(contents)
    (stage / "data").mkdir(exist_ok=True)
    for rel in DATA:
        shutil.copy2(prepared / rel, stage / rel)
    for rel, wanted in frozen_manifest.items():
        if sha(stage / rel) != wanted:
            raise RuntimeError("Staged official source mismatch: " + rel)
    env["PYTHONPATH"] = str(stage)
    # Inventory subprocess exits before preflight/measurement: no persistent CUDA context.
    inventory = command([sys.executable, str(helper / "check_kaggle_environment.py"), "--gpu",
                         "--require-four-cpus", "--out", str(runroot / "environment.json")],
                        stage, env, runroot / "inventory.log", 120)
    if inventory["exit_code"] != 0:
        raise RuntimeError("Pinned environment/T4 check failed; see " + inventory["log"])
    current = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
    if sum(int(line) for line in current.splitlines()) >= 1000:
        raise RuntimeError("GPU idle-start requirement not met")
    manifest = {name: hashlib.sha256(text.encode()).hexdigest() for name, text in payload.items() if name.startswith("solution/")}
    dump(runroot / "executed_source_manifest.json", manifest)
    dump(runroot / "cohort.json", {"backend": config["backend"], "official_commit": COMMIT,
         "seed": 1, "repeats_each": 2, "order": ["base", "solution", "solution", "base"],
         "native_timeout_s": 900, "outer_timeout_s": 10800, "baseline_id": None,
         "baseline_note": "Own paired baseline, no inherited Kaggle79s limit; not a registered queue baseline",
         "score": None, "model_manifest_sha256": sha(runroot / "model_manifest.json")})
    # Fixed, bounded fixture; no tuning. Its caches are isolated and never reused by measurements.
    pre_env = dict(env, TRITON_CACHE_DIR=str(runroot / "preflight_triton"),
                   TORCHINDUCTOR_CACHE_DIR=str(runroot / "preflight_inductor"))
    preflight = command([sys.executable, str(helper / "gpu_preflight.py"), "--out", str(runroot / "preflight.json")],
                        stage, pre_env, runroot / "preflight.log", 600)
    if preflight["exit_code"] != 0 or not (runroot / "preflight.json").is_file():
        dump(runroot / "summary.json", {"status": "PREFLIGHT_FAILED_OR_INCOMPLETE", "score": None,
              "admission": "UNKNOWN", "preflight": preflight})
        raise RuntimeError("Exact preflight failed/incomplete; measured pair not started")
    # The preflight subprocess has exited; recheck idle now, not only before it.
    current = subprocess.check_output(["nvidia-smi", "--query-gpu=memory.used", "--format=csv,noheader,nounits"], text=True)
    if sum(int(line) for line in current.splitlines()) >= 1000:
        raise RuntimeError("Post-preflight GPU idle-start requirement not met")
    # Unmodified official runner provides fresh child process/model/adapter/optimizer/output
    # and empty Triton/Inductor directories for every measured side and repetition.
    output = runroot / "measured"
    receipt = command([sys.executable, "harness/run_task.py", "--repeats", "2", "--seed", "1",
                       "--timeout", "900", "--out", str(output)], stage, env,
                      runroot / "paired.log", 10800)
    result_path = output / "result.json"
    strict_complete = False
    checker = None
    details = []
    if result_path.is_file():
        result = json.loads(result_path.read_text())
        for side in ("base", "solution"):
            runs = result.get("runs", {}).get(side, [])
            side_ok = len(runs) == 2
            for r in runs:
                m = r.get("metrics", {})
                ok = (r.get("exit_code") == 0 and m.get("exit_code") == 0
                      and r.get("global_step") == 64 and m.get("steps") == 64
                      and bool(r.get("adapter_tensors")) and bool(r.get("adapter_config")))
                side_ok = side_ok and ok
                details.append({"side": side, "run": r.get("run"), "complete": bool(ok),
                      "command_wall_s": r.get("wall_s"), "external_peak_mib": r.get("gpu_peak_mib"),
                      "own_kernel_ms": m.get("own_kernel_ms"), "timed_loop_s": m.get("timed_loop_s"),
                      "triton_launches": m.get("triton_launches")})
            ev = result.get("evals", {}).get(side, {})
            loss = ev.get("mean_loss")
            side_ok = side_ok and ev.get("exit_code") == 0 and isinstance(loss, (int, float)) and math.isfinite(loss)
            if side == "base":
                strict_complete = side_ok
            else:
                strict_complete = strict_complete and side_ok
        strict_complete = bool(strict_complete and receipt["exit_code"] == 0 and not receipt["outer_timeout"])
        # Even unsuccessful result gets truthful official checker output, not a score.
        checker = command([sys.executable, "harness/check_rules.py", "--result", str(result_path)],
                          stage, env, runroot / "checks.log", 900)
    activation = []
    for log in output.glob("solution_*/train.log"):
        activation.extend({"log": str(log), "receipt": line.strip()} for line in log.read_text(errors="replace").splitlines()
                          if line.startswith("SOUP_NF4_ROUTE "))
    dump(runroot / "activation_receipts.json", activation)
    checks = json.loads((output / "checks.json").read_text()) if (output / "checks.json").exists() else None
    summary = {"status": "COMPLETE_RESEARCH_PAIR" if strict_complete else "FAILED_PARTIAL_OR_TIMEOUT",
               "complete": strict_complete, "score": None, "admission": "UNKNOWN",
               "backend": config["backend"], "runs": details, "outer": receipt, "checker": checker,
               "result_sha256": sha(result_path) if result_path.exists() else None,
               "checks_sha256": sha(output / "checks.json") if checks is not None else None,
               "activation_receipt_count": len(activation),
               "all_10_preliminary_checks_passed": bool(strict_complete and checker and checker["exit_code"] == 0 and checks and len(checks.get("checks", [])) == 10 and all(c.get("ok") is True for c in checks["checks"])),
               "limits": "Official checks are preliminary; no hidden quality or final Kaggle admission. GCP is its own cohort."}
    dump(runroot / "summary.json", summary)
    print(json.dumps(summary, indent=2))
    if not strict_complete:
        raise RuntimeError("Attempt incomplete; all evidence retained, score remains null: " + str(runroot))
    return runroot


In [ ]:
RUN_DIRECTORY = run(CONFIG, PAYLOAD, FROZEN_MANIFEST)
print("Evidence directory:", RUN_DIRECTORY)


## Evidence boundaries
CPU interpreter:48 fixtures passed; local Torch2.15 CPU/Triton3.9 is not target Torch2.11/Triton3.6.
Four offline SM75 compilations succeeded with zero shared bytes, separate FP32 multiply/add and
packed32-bit stores where requested. No GPU code was executed and no compiled binaries are embedded.
T4 numerical parity, actual route coverage, full task quality, speed, peak and≥5% own share are unknown.
Never sum historical shares or slow/add work to meet the share threshold. Inspect failed/partial logs;
they receive neither scores nor admission. Raw measured own-share comes only from official events
following two warmup steps. Flushed route logs are activation receipts, not timing evidence.
